In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import csv
import pip
import seaborn as sns
import matplotlib.pyplot as plt
from functools import reduce
import statsmodels.api as sm
import linearmodels as lm
from linearmodels import PanelOLS, RandomEffects
from scipy import stats
from linearmodels import RandomEffects
import statsmodels.api as sm
from linearmodels.panel import compare

In [ ]:
final_df = pd.read_csv("prepared_data_for_regression.csv")
print(final_df.head(5))

In [ ]:
initial_countries=final_df['country'].nunique()
print(initial_countries)

In [ ]:
initial_years=final_df['year'].nunique()
print(initial_years)

# OECD  Countries :

In [ ]:
oecd_countries = ['austria', 'australia', 'belgium', 'canada', 'chile', 
                  'colombia', 'czech republic', 'denmark', 'estonia', 'finland', 
                  'france', 'germany', 'greece', 'hungary', 'iceland', 'ireland', 
                  'israel', 'italy', 'japan', 'korea', 'latvia', 'lithuania', 'luxembourg', 
                  'mexico', 'netherlands', 'new zealand', 'norway', 'poland', 'portugal', 'slovak republic', 
                  'slovenia', 'spain', 'sweden', 'switzerland', 'turkey', 'united kingdom', 'united states', 'costa rica']

# Creat Dummy Variables:

In [ ]:
#final_df['is_oecd'] = final_df['country'].isin(oecd_list).astype(int)
final_df['is_oecd'] = final_df['country'].isin(oecd_countries).astype(int)

# creat interaction :

In [ ]:
final_df['bmi_x_oecd'] = final_df['bmi']* final_df['is_oecd']
print(final_df['is_oecd'].value_counts())
columns_for_heatmap=['bmi', 'gdp', 'urban_pop', 'bmi_x_oecd', 'is_oecd', 'labor_rate', 'pop_65'
                      ,'incidence', 'mortality', 'life_exp', 'fertility']
correlation_matrix = final_df[columns_for_heatmap].corr()
plt.figure(figsize= (12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", center=0, linewidths=0.5)
plt.title('correlation matrix heatmap')
plt.show()

# Dummy Variables: 

In [ ]:
final_df['is_oecd']= final_df['country'].str.lower().str.strip().isin(oecd_countries).astype(int)
print(final_df['is_oecd'].value_counts())

In [ ]:
oecd_comparison= final_df.groupby('is_oecd').incidence.mean()
print(final_df['is_oecd'].value_counts())

1. Aging Society Dummy: Based on WHO standards, societies with >10% population over 65 are considered "ageing" or "aged". This captures potential non-linear increases in cancer incidence due to population structure.

In [ ]:
final_df['dm_high_aging_society'] = (final_df['pop_65'] > 10).astype(int)
print(final_df['dm_high_aging_society'].value_counts())

2. High Life Expectancy Dummy threshold of 75 years represents advanced healthcare systems and higher probability of disease detection.

In [ ]:
final_df['dm_high_life_exp']=(final_df['life_exp']> 75).astype(int)
print(final_df['dm_high_life_exp'].value_counts())

In [ ]:
comparison_inc= final_df.groupby('dm_high_life_exp').incidence.mean()
print(comparison_inc)

In [ ]:
comparison_mortal= final_df.groupby('dm_high_life_exp').mortality.mean()
print(comparison_mortal)

In [ ]:
print(final_df[['year', 'country', 'is_oecd', 'dm_high_aging_society', 'dm_high_life_exp']].head(100))

# create log gdp

In [ ]:
final_df['log_gdp']= np.log(final_df['gdp'])

In [ ]:
final_df['log_gdp_is_oecd']= final_df['log_gdp']*final_df['is_oecd']
print(final_df['log_gdp_is_oecd'].head(5))

# interaction of is_oecd variable and gdp

In [ ]:
final_df['log_gdp_is_oecd']= final_df['gdp']*final_df['is_oecd']
print(final_df['log_gdp_is_oecd'].tail(5))

In [ ]:
print(final_df[['year', 'country', 'log_gdp_is_oecd', 'dm_high_aging_society', 'dm_high_life_exp']].tail(100))

In [ ]:
start_year=final_df['year'].min()
end_year=final_df['year'].max()
print(final_df[['year', 'country', 'dm_high_aging_society', 'dm_high_life_exp']].tail(100))
start_count= final_df[final_df['year']== start_year]['dm_high_life_exp'].sum() 
end_count=final_df[final_df['year']== end_year]['dm_high_life_exp'].sum()
print(f"number of countries with high life_expectency in {start_count}")
print(f"number of countries with high life_expectency in {end_count}")

# delet rows with missing value(NA) in all variables 

In [ ]:
print(f" Missing values in final_df: {final_df.isnull().sum()}")

In [ ]:
clean_df= final_df.dropna(subset=['incidence','mortality', 'gdp', 'pop_65', 'urban_pop', 'life_exp', 'fertility', 'labor_rate', 'log_gdp_is_oecd'])
print(f" Missing values after deleting NA in clean_df: {clean_df.isnull().sum()}")

# Building stepwise regression

# Step 1: Adding Urban Population and log_gdp

In [ ]:
df_step= clean_df.set_index(['country', 'year'])
exog_1_fe= sm.add_constant(df_step[['log_gdp', 'urban_pop']])
model_1_fe=PanelOLS(df_step['incidence'],exog_1_fe, entity_effects=True, time_effects=True).fit()
print(model_1_fe.summary)

# Step 2: Adding Pop_65:

In [ ]:
exog_2_fe = df_step[['log_gdp', 'urban_pop', 'pop_65']]
model_2_fe = PanelOLS(df_step['incidence'], exog_2_fe, entity_effects=True, time_effects=True).fit()
print(model_2_fe.summary)

# Step 3: Adding Labor Rate:

In [ ]:
exog_3_fe=sm.add_constant(df_step[['log_gdp', 'urban_pop', 'pop_65', 'labor_rate']])
model_3_fe=PanelOLS(df_step['incidence'], exog_3_fe, entity_effects=True, time_effects=True).fit()
print(model_3_fe.summary)

# Step 4: Adding BMI:

In [ ]:
exog_4_fe=sm.add_constant(df_step[['log_gdp','urban_pop', 'pop_65', 'labor_rate', 'bmi']])
model_4_fe=PanelOLS(df_step['incidence'], exog_4_fe, entity_effects=True, time_effects= True).fit()
print(model_4_fe.summary)

# Step 5: Adding is_oecd as Dummy Variable:

In [ ]:
exog_5_fe=sm.add_constant(df_step[['log_gdp','urban_pop', 'pop_65', 'labor_rate', 'bmi','log_gdp_is_oecd']])
model_5_fe=PanelOLS(df_step['incidence'], exog_5_fe, entity_effects=True, time_effects=True).fit()
print(model_5_fe.summary)

# Step 6: Adding High aging society as Dummy Variable:

In [ ]:
exog_6_fe = sm.add_constant(df_step[['log_gdp', 'urban_pop', 'pop_65', 'labor_rate', 'bmi','log_gdp_is_oecd', 'dm_high_aging_society']])
model_6_fe = PanelOLS(df_step['incidence'], exog_6_fe, entity_effects=True, time_effects=True).fit()
print(model_6_fe.summary)

# Step 7: Adding High life exp as Dummy Variable:

In [ ]:
exog_7_fe = sm.add_constant(df_step[['log_gdp', 'urban_pop', 'pop_65', 'labor_rate', 'bmi','log_gdp_is_oecd', 'dm_high_aging_society','dm_high_life_exp']])
model_7_fe = PanelOLS(df_step['incidence'], exog_7_fe, entity_effects=True, time_effects= True).fit()
print(model_7_fe.summary)

# create labor_rate * is_oecd variable:

In [65]:
df_step['labor_rate * is_oecd']= df_step['labor_rate']* df_step['is_oecd']

# Step 8: Adding labor_rate * is_oecd:

In [66]:
exog_8_fe = sm.add_constant(df_step[['log_gdp', 'urban_pop', 'pop_65', 'labor_rate', 'bmi','log_gdp_is_oecd', 'dm_high_aging_society','dm_high_life_exp', 'labor_rate * is_oecd']])
model_8_fe = PanelOLS(df_step['incidence'], exog_8_fe, entity_effects=True, time_effects= True).fit()
print(model_8_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:              incidence   R-squared:                        0.2553
Estimator:                   PanelOLS   R-squared (Between):              0.3132
No. Observations:                5154   R-squared (Within):               0.4665
Date:                Fri, Feb 27 2026   R-squared (Overall):              0.3183
Time:                        15:33:55   Log-likelihood                -1.692e+04
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      188.86
Entities:                         153   P-value                           0.0000
Avg Obs:                       33.686   Distribution:                  F(9,4959)
Min Obs:                       11.000                                           
Max Obs:                       34.000   F-statistic (robust):             188.86
                            

# Comparing results of all Fixed Effect models:

In [67]:
Compare_results_fixed_Effect = compare({
    'FE model 1': model_1_fe,
    'FE model 2': model_2_fe,
    'FE model 3': model_3_fe,
    'FE model 4': model_4_fe,
    'FE model 5': model_5_fe,
    'FE model 6': model_6_fe,
    'FE model 7': model_7_fe,
    'FE model 8': model_8_fe
    })

print(Compare_results_fixed_Effect)

                                                                 Model Comparison                                                                
                              FE model 1     FE model 2     FE model 3     FE model 4     FE model 5     FE model 6     FE model 7     FE model 8
-------------------------------------------------------------------------------------------------------------------------------------------------
Dep. Variable                  incidence      incidence      incidence      incidence      incidence      incidence      incidence      incidence
Estimator                       PanelOLS       PanelOLS       PanelOLS       PanelOLS       PanelOLS       PanelOLS       PanelOLS       PanelOLS
No. Observations                    5154           5154           5154           5154           5154           5154           5154           5154
Cov. Est.                     Unadjusted     Unadjusted     Unadjusted     Unadjusted     Unadjusted     Unadjusted     Unad